In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)

print("Environment loaded successfully.")

Environment loaded successfully.


In [2]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/baseline.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.baseline import DATA_PATH, TABLES, load_split, build_model, save_pipeline, CATEGORICAL, NUMERIC

from src.baseline import RAW_PATH, clean_data, prepare_data
DATA_PATH = RAW_PATH

df = pd.read_excel(DATA_PATH, header=1)

print("Shape:", df.shape)
df.head()

Shape: (30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [3]:
print("ROWS:", df.shape[0])
print("COLUMNS:", df.shape[1])

print("\nCOLUMN NAMES:")
print(df.columns.tolist())

print("\nDATA TYPES:")
print(df.dtypes)

print("\nMISSING VALUES:")
print(df.isna().sum())

print("\nDUPLICATE ROWS:")
print("Excluding ID, including outcome:", df.drop(columns="ID").duplicated().sum())
print("Records involved:", df.drop(columns="ID").duplicated(keep=False).sum())

ROWS: 30000
COLUMNS: 25

COLUMN NAMES:
['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default payment next month']

DATA TYPES:
ID                            int64
LIMIT_BAL                     int64
SEX                           int64
EDUCATION                     int64
MARRIAGE                      int64
AGE                           int64
PAY_0                         int64
PAY_2                         int64
PAY_3                         int64
PAY_4                         int64
PAY_5                         int64
PAY_6                         int64
BILL_AMT1                     int64
BILL_AMT2                     int64
BILL_AMT3                     int64
BILL_AMT4                     int64
BILL_AMT5                     int64
BILL_AMT6                     int64
PAY_AMT

In [4]:
for col in ["SEX", "EDUCATION", "MARRIAGE"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).sort_index())


--- SEX ---
SEX
1    11888
2    18112
Name: count, dtype: int64

--- EDUCATION ---
EDUCATION
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51
Name: count, dtype: int64

--- MARRIAGE ---
MARRIAGE
0       54
1    13659
2    15964
3      323
Name: count, dtype: int64


In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
ID,30000.0,15000.500000,8660.398374,1.0,7500.75,15000.5,22500.25,30000.0
LIMIT_BAL,30000.0,167484.322667,129747.661567,10000.0,50000.00,140000.0,240000.00,1000000.0
SEX,30000.0,1.603733,0.489129,1.0,1.00,2.0,2.00,2.0
EDUCATION,30000.0,1.853133,0.790349,0.0,1.00,2.0,2.00,6.0
MARRIAGE,30000.0,1.551867,0.521970,0.0,1.00,2.0,2.00,3.0
AGE,30000.0,35.485500,9.217904,21.0,28.00,34.0,41.00,79.0
PAY_0,30000.0,-0.016700,1.123802,-2.0,-1.00,0.0,0.00,8.0
PAY_2,30000.0,-0.133767,1.197186,-2.0,-1.00,0.0,0.00,8.0
PAY_3,30000.0,-0.166200,1.196868,-2.0,-1.00,0.0,0.00,8.0
PAY_4,30000.0,-0.220667,1.169139,-2.0,-1.00,0.0,0.00,8.0


In [6]:
target = "default payment next month"

print(df[target].value_counts())
print("\nPercentages:")
print(df[target].value_counts(normalize=True) * 100)

default payment next month
0    23364
1     6636
Name: count, dtype: int64

Percentages:
default payment next month
0    77.88
1    22.12
Name: proportion, dtype: float64


Data Cleaning

In [7]:
df_clean = clean_data(df)
# ID is retained for traceability and explicitly excluded by both models.
print("Clean shape:", df_clean.shape)
df_clean.head()

Clean shape:

 (30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [8]:
df_clean, split_metadata, duplicate_audit = prepare_data()
print("Saved cleaned data and shared train/test metadata.")
duplicate_audit

Saved cleaned data and shared train/test metadata.


{'raw_all_except_ID': {'excess_duplicate_rows': 35,
  'records_in_duplicate_groups': 70,
  'duplicate_groups': 35,
  'original_crossing_groups': 7,
  'original_train_records_in_crossing_groups': 7,
  'original_test_records_in_crossing_groups': 7},
 'clean_all_except_ID': {'excess_duplicate_rows': 35,
  'records_in_duplicate_groups': 70,
  'duplicate_groups': 35,
  'original_crossing_groups': 7,
  'original_train_records_in_crossing_groups': 7,
  'original_test_records_in_crossing_groups': 7},
 'clean_predictors_including_SEX': {'excess_duplicate_rows': 56,
  'records_in_duplicate_groups': 108,
  'duplicate_groups': 52,
  'original_crossing_groups': 15,
  'original_train_records_in_crossing_groups': 16,
  'original_test_records_in_crossing_groups': 15},
 'model_predictors_excluding_ID_SEX_target': {'excess_duplicate_rows': 93,
  'records_in_duplicate_groups': 177,
  'duplicate_groups': 84,
  'original_crossing_groups': 31,
  'original_train_records_in_crossing_groups': 33,
  'original_t